In [ ]:
# Copyright (c) 2025 Steven P.LI
# https://github.com/StvLi/
#
# This code is licensed under the MIT License.
# See the LICENSE file in the repository for the full license text.

from typing import TypedDict

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

# --- Configuration ---
# Ensure your API key environment variable is set (e.g., GOOGLE_API_KEY)
try:
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
    print(f"Language model initialized: {llm.model}")
except Exception as e:
    print(f"Error initializing language model: {e}")
    llm = None

# --- Define Simulated Sub-Agent Handlers (equivalent to ADK sub_agents) ---

def booking_handler(request: str) -> str:
    """Simulates the Booking Agent handling a request."""
    print("\n--- DELEGATING TO BOOKING HANDLER ---")
    return f"Booking Handler processed request: '{request}'. Result: Simulated booking action."

def info_handler(request: str) -> str:
    """Simulates the Info Agent handling a request."""
    print("\n--- DELEGATING TO INFO HANDLER ---")
    return f"Info Handler processed request: '{request}'. Result: Simulated information retrieval."

def unclear_handler(request: str) -> str:
    """Handles requests that couldn't be delegated."""
    print("\n--- HANDLING UNCLEAR REQUEST ---")
    return f"Coordinator could not delegate request: '{request}'. Please clarify."

# --- Define the Graph State ---
# The state travels through every node. Each node returns only the keys it writes.

class RouterState(TypedDict, total=False):
    request: str    # the original user request, never mutated
    decision: str   # one of 'booker', 'info', 'unclear'
    output: str     # the final answer produced by the selected handler

# --- Define the Coordinator Router Chain (the classifier) ---
# This is the only place in the graph that calls the language model.

coordinator_router_prompt = ChatPromptTemplate.from_messages([
    ("system", """Analyze the user's request and determine which specialist handler should process it.
     - If the request is related to booking flights or hotels, output 'booker'.
     - For all other general information questions, output 'info'.
     - If the request is unclear or doesn't fit either category, output 'unclear'.
     ONLY output one word: 'booker', 'info', or 'unclear'."""),
    ("user", "{request}")
])

if llm:
    coordinator_router_chain = coordinator_router_prompt | llm | StrOutputParser()

# --- Define the Graph Nodes ---

def router_node(state: RouterState) -> dict:
    """Classify the request and write the decision into the state."""
    decision = coordinator_router_chain.invoke({"request": state["request"]})
    decision = decision.strip().lower()
    print(f"\n--- ROUTER DECISION: '{decision}' ---")
    return {"decision": decision}

def booking_node(state: RouterState) -> dict:
    return {"output": booking_handler(state["request"])}

def info_node(state: RouterState) -> dict:
    return {"output": info_handler(state["request"])}

def unclear_node(state: RouterState) -> dict:
    return {"output": unclear_handler(state["request"])}

# --- Define the Conditional Edge ---
# Returns the name of the next node. Anything unexpected falls back to 'unclear'.

def route_decision(state: RouterState) -> str:
    decision = state.get("decision", "")
    if decision in ("booker", "info"):
        return decision
    return "unclear"

# --- Build the Graph ---

def build_graph():
    graph = StateGraph(RouterState)

    graph.add_node("router", router_node)
    graph.add_node("booker", booking_node)
    graph.add_node("info", info_node)
    graph.add_node("unclear", unclear_node)

    graph.add_edge(START, "router")
    graph.add_conditional_edges(
        "router",
        route_decision,
        {"booker": "booker", "info": "info", "unclear": "unclear"},
    )
    graph.add_edge("booker", END)
    graph.add_edge("info", END)
    graph.add_edge("unclear", END)

    return graph.compile()

# --- Example Usage ---

def main():
    if not llm:
        print("\nSkipping execution due to LLM initialization failure.")
        return

    coordinator_agent = build_graph()

    print("--- Running with a booking request ---")
    request_a = "Book me a flight to London."
    result_a = coordinator_agent.invoke({"request": request_a})["output"]
    print(f"Final Result A: {result_a}")

    print("\n--- Running with an info request ---")
    request_b = "What is the capital of Italy?"
    result_b = coordinator_agent.invoke({"request": request_b})["output"]
    print(f"Final Result B: {result_b}")

    print("\n--- Running with an unclear request ---")
    request_c = "Tell me about quantum physics."
    result_c = coordinator_agent.invoke({"request": request_c})["output"]
    print(f"Final Result C: {result_c}")

if __name__ == "__main__":
    main()
